# Linear Model: SGDClassifier con HashingVectorizer

En este notebook se entrena un modelo lineal basado en `SGDClassifier` con función de pérdida `log_loss`.

La representación textual se construye mediante `HashingVectorizer`, combinando n-gramas de palabras y n-gramas de caracteres. Esta técnica permite trabajar con vectores de alta dimensión de forma eficiente en memoria.

Además, se añaden variables numéricas simples del texto, como la longitud de la afirmación, el número de palabras y el número de temas. El ajuste de hiperparámetros se realiza mediante `RandomizedSearchCV`, y posteriormente se ajusta el threshold para maximizar F1 macro.

In [1]:
import pandas as pd
import numpy as np

from scipy.stats import loguniform, uniform

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import HashingVectorizer, TfidfTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDClassifier

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import warnings
warnings.filterwarnings("ignore")

In [2]:
train_prep = pd.read_csv("../data/train_preprocessed.csv")
test_prep = pd.read_csv("../data/test_preprocessed.csv")

print("Train:", train_prep.shape)
print("Test:", test_prep.shape)

display(train_prep.head())
display(test_prep.head())

Train: (8950, 26)
Test: (3836, 25)


,id,label,statement,subject,speaker,speaker_job,state_info,party_affiliation,statement_clean,statement_length,...,speaker_grouped,speaker_grouped_clean,speaker_job_clean,speaker_job_freq,speaker_job_grouped,speaker_job_grouped_clean,state_info_clean,state_info_freq,party_affiliation_clean,party_affiliation_freq
0,81f884c64a7,1,China is in the South China Sea and (building)...,"china,foreign-policy,military",donald-trump,President-Elect,New York,republican,china is in the south china sea and buildinga ...,116,...,donald-trump,donald trump,president elect,247,President-Elect,president elect,new york,579,republican,3947
1,30c2723a188,0,With the resources it takes to execute just ov...,health-care,chris-dodd,U.S. senator,Connecticut,democrat,with the resources it takes to execute just ov...,164,...,chris-dodd,chris dodd,u s senator,236,U.S. senator,u s senator,connecticut,21,democrat,2898
2,6936b216e5d,0,The (Wisconsin) governor has proposed tax give...,"corporations,pundits,taxes,abc-news-week",donna-brazile,Political commentator,"Washington, D.C.",democrat,the wisconsin governor has proposed tax giveaw...,68,...,donna-brazile,donna brazile,political commentator,15,Political commentator,political commentator,washington d c,89,democrat,2898
3,b5cd9195738,1,Says her representation of an ex-boyfriend who...,"candidates-biography,children,ethics,families,...",rebecca-bradley,unknown,unknown,none,says her representation of an exboyfriend who ...,135,...,other,other,unknown,2482,unknown,unknown,unknown,1930,none,1531
4,84f8dac7737,0,At protests in Wisconsin against proposed coll...,"health-care,labor,state-budget",republican-party-wisconsin,unknown,Wisconsin,republican,at protests in wisconsin against proposed coll...,141,...,republican-party-wisconsin,republican party wisconsin,unknown,2482,unknown,unknown,wisconsin,648,republican,3947


,id,statement,subject,speaker,speaker_job,state_info,party_affiliation,statement_clean,statement_length,statement_word_count,...,speaker_grouped,speaker_grouped_clean,speaker_job_clean,speaker_job_freq,speaker_job_grouped,speaker_job_grouped_clean,state_info_clean,state_info_freq,party_affiliation_clean,party_affiliation_freq
0,dc32e5ffa8b,Five members of [the Common Cause Georgia] boa...,"campaign-finance,ethics,government-regulation",kasim-reed,unknown,unknown,democrat,five members of the common cause georgia board...,89,12,...,kasim-reed,kasim reed,unknown,2482.0,unknown,unknown,unknown,1930.0,democrat,2898
1,aa49bb41cab,Theres no negative advertising in my campaign ...,elections,bill-mccollum,unknown,Florida,republican,theres no negative advertising in my campaign ...,53,9,...,bill-mccollum,bill mccollum,unknown,2482.0,unknown,unknown,florida,853.0,republican,3947
2,dddc8d12ac1,Leticia Van de Putte voted to give illegal imm...,"health-care,immigration,public-health",dan-patrick,Lieutenant governor-elect,Texas,republican,leticia van de putte voted to give illegal imm...,141,23,...,dan-patrick,dan patrick,lieutenant governor elect,18.0,Lieutenant governor-elect,lieutenant governor elect,texas,879.0,republican,3947
3,bcfe8f51667,Fiorinas plan would mean slashing Social Secur...,"federal-budget,medicare,social-security",barbara-boxer,U.S. Senator,California,democrat,fiorinas plan would mean slashing social secur...,63,9,...,barbara-boxer,barbara boxer,u s senator,391.0,U.S. Senator,u s senator,california,121.0,democrat,2898
4,eedbbaff5ab,"By the end of his first term, President Obama ...","federal-budget,new-hampshire-2012",mitt-romney,Former governor,Massachusetts,republican,by the end of his first term president obama w...,115,22,...,mitt-romney,mitt romney,former governor,142.0,Former governor,former governor,massachusetts,167.0,republican,3947


In [3]:
text_cols = [
    "statement_clean",
    "subject_clean",
    "speaker_grouped_clean",
    "speaker_job_grouped_clean",
    "state_info_clean",
    "party_affiliation_clean"
]

for col in text_cols:
    train_prep[col] = train_prep[col].fillna("unknown").astype(str)
    test_prep[col] = test_prep[col].fillna("unknown").astype(str)

In [4]:
train_prep["text_features_sgd"] = (
    train_prep["statement_clean"] + " " +
    train_prep["subject_clean"] + " " +
    train_prep["speaker_grouped_clean"] + " " +
    train_prep["speaker_job_grouped_clean"] + " " +
    train_prep["state_info_clean"] + " " +
    train_prep["party_affiliation_clean"]
)

test_prep["text_features_sgd"] = (
    test_prep["statement_clean"] + " " +
    test_prep["subject_clean"] + " " +
    test_prep["speaker_grouped_clean"] + " " +
    test_prep["speaker_job_grouped_clean"] + " " +
    test_prep["state_info_clean"] + " " +
    test_prep["party_affiliation_clean"]
)

display(train_prep[["text_features_sgd", "label"]].head())

,text_features_sgd,label
0,china is in the south china sea and buildinga ...,1
1,with the resources it takes to execute just ov...,0
2,the wisconsin governor has proposed tax giveaw...,0
3,says her representation of an exboyfriend who ...,1
4,at protests in wisconsin against proposed coll...,0


In [5]:
numeric_features = [
    "statement_length",
    "statement_word_count",
    "num_subjects"
]

train_prep[numeric_features] = train_prep[numeric_features].fillna(0)
test_prep[numeric_features] = test_prep[numeric_features].fillna(0)

In [6]:
X = train_prep[["text_features_sgd"] + numeric_features]
y = train_prep["label"]

X_test = test_prep[["text_features_sgd"] + numeric_features]

print("X:", X.shape)
print("y:", y.shape)
print("X_test:", X_test.shape)

X: (8950, 4)
y: (8950,)
X_test: (3836, 4)


In [7]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape)
print("Valid:", X_valid.shape)

Train: (7160, 4)
Valid: (1790, 4)


In [8]:
word_pipeline = Pipeline([
    ("hash", HashingVectorizer(
        alternate_sign=False,
        norm=None,
        analyzer="word",
        ngram_range=(1, 2),
        lowercase=False
    )),
    ("tfidf", TfidfTransformer())
])

char_pipeline = Pipeline([
    ("hash", HashingVectorizer(
        alternate_sign=False,
        norm=None,
        analyzer="char_wb",
        ngram_range=(3, 5),
        lowercase=False
    )),
    ("tfidf", TfidfTransformer())
])

text_vectorizer = FeatureUnion([
    ("word", word_pipeline),
    ("char", char_pipeline)
])

In [9]:
preprocessor = ColumnTransformer(
    transformers=[
        ("text", text_vectorizer, "text_features_sgd"),
        ("num", StandardScaler(), numeric_features)
    ]
)

In [10]:
sgd = SGDClassifier(
    loss="log_loss",
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=5
)

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("clf", sgd)
])

In [11]:
param_distributions = {
    "preprocessor__text__word__hash__n_features": [2**18, 2**19, 2**20],
    "preprocessor__text__char__hash__n_features": [2**16, 2**17, 2**18],
    "clf__alpha": loguniform(1e-7, 1e-3),
    "clf__penalty": ["l2", "l1", "elasticnet"],
    "clf__l1_ratio": uniform(0.05, 0.9),
    "clf__learning_rate": ["optimal", "adaptive"],
    "clf__eta0": loguniform(1e-4, 1e-1),
    "clf__max_iter": [1000, 1500, 2000],
    "clf__class_weight": [None, "balanced"]
}

random_search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_distributions,
    n_iter=40,
    scoring="f1_macro",
    cv=3,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

Fitting 3 folds for each of 40 candidates, totalling 120 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'clf__alpha': <scipy.stats....00188E99AC3D0>, 'clf__class_weight': [None, 'balanced'], 'clf__eta0': <scipy.stats....00188E99AD910>, 'clf__l1_ratio': <scipy.stats....00188E99AD450>, ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",40
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'f1_macro'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the v

In [12]:
print("Mejor F1 macro CV:", random_search.best_score_)
print("Mejores parámetros:")
print(random_search.best_params_)

Mejor F1 macro CV: 0.6025620962090928
Mejores parámetros:
{'clf__alpha': np.float64(1.0842262717330158e-06), 'clf__class_weight': 'balanced', 'clf__eta0': np.float64(0.018453732926615934), 'clf__l1_ratio': np.float64(0.4326402870421202), 'clf__learning_rate': 'adaptive', 'clf__max_iter': 1500, 'clf__penalty': 'l1', 'preprocessor__text__char__hash__n_features': 65536, 'preprocessor__text__word__hash__n_features': 1048576}


In [13]:
best_sgd_model = random_search.best_estimator_

valid_proba = best_sgd_model.predict_proba(X_valid)[:, 1]
valid_pred_05 = (valid_proba >= 0.5).astype(int)

print("Threshold 0.5")
print("Accuracy:", accuracy_score(y_valid, valid_pred_05))
print("F1 macro:", f1_score(y_valid, valid_pred_05, average="macro"))
print(classification_report(y_valid, valid_pred_05))
print(confusion_matrix(y_valid, valid_pred_05))

Threshold 0.5
Accuracy: 0.6273743016759776
F1 macro: 0.6121179633374756
              precision    recall  f1-score   support

           0       0.48      0.61      0.54       631
           1       0.75      0.64      0.69      1159

    accuracy                           0.63      1790
   macro avg       0.61      0.62      0.61      1790
weighted avg       0.65      0.63      0.63      1790

[[384 247]
 [420 739]]


In [14]:
thresholds = np.arange(0.20, 0.80, 0.005)

threshold_results = []

for threshold in thresholds:
    valid_pred = (valid_proba >= threshold).astype(int)
    
    threshold_results.append({
        "threshold": threshold,
        "accuracy": accuracy_score(y_valid, valid_pred),
        "f1_macro": f1_score(y_valid, valid_pred, average="macro"),
        "f1_class_1": f1_score(y_valid, valid_pred)
    })

threshold_results = pd.DataFrame(threshold_results)

display(threshold_results.sort_values("f1_macro", ascending=False).head(15))

best_threshold = threshold_results.sort_values("f1_macro", ascending=False).iloc[0]["threshold"]
print("Mejor threshold:", best_threshold)

,threshold,accuracy,f1_macro,f1_class_1
59,0.495,0.629609,0.612225,0.694329
60,0.500,0.627374,0.612118,0.689044
55,0.475,0.637989,0.611427,0.713020
58,0.490,0.630726,0.611215,0.698311
57,0.485,0.632402,0.611200,0.701993
56,0.480,0.632961,0.609013,0.705777
61,0.505,0.622346,0.608723,0.681733
54,0.470,0.636313,0.606198,0.715098
62,0.510,0.617877,0.605585,0.675214
64,0.520,0.613408,0.604407,0.664078


Mejor threshold: 0.4950000000000003


In [15]:
valid_pred_best = (valid_proba >= best_threshold).astype(int)

print("Accuracy:", accuracy_score(y_valid, valid_pred_best))
print("F1 macro:", f1_score(y_valid, valid_pred_best, average="macro"))
print("F1 clase 1:", f1_score(y_valid, valid_pred_best))

print(classification_report(y_valid, valid_pred_best))
print(confusion_matrix(y_valid, valid_pred_best))

Accuracy: 0.629608938547486
F1 macro: 0.6122248329417255
F1 clase 1: 0.69432918395574
              precision    recall  f1-score   support

           0       0.48      0.59      0.53       631
           1       0.75      0.65      0.69      1159

    accuracy                           0.63      1790
   macro avg       0.61      0.62      0.61      1790
weighted avg       0.65      0.63      0.64      1790

[[374 257]
 [406 753]]


In [16]:
best_sgd_model.fit(X, y)

test_proba = best_sgd_model.predict_proba(X_test)[:, 1]

In [17]:
thresholds_to_try = [
    round(float(best_threshold), 3),
    0.40,
    0.45,
    0.48,
    0.50,
    0.52,
    0.55
]

thresholds_to_try = sorted(set(thresholds_to_try))

for threshold in thresholds_to_try:
    test_predictions = (test_proba >= threshold).astype(int)
    
    submission = pd.DataFrame({
        "id": test_prep["id"],
        "label": test_predictions
    })
    
    filename = f"submission_sgd_hashing_threshold_{str(threshold).replace('.', '')}.csv"
    submission.to_csv(filename, index=False)
    
    print("\n==============================")
    print("Archivo:", filename)
    print("Threshold:", threshold)
    print("Distribución:")
    print(submission["label"].value_counts())


Archivo: submission_sgd_hashing_threshold_04.csv
Threshold: 0.4
Distribución:
label
1    3000
0     836
Name: count, dtype: int64

Archivo: submission_sgd_hashing_threshold_045.csv
Threshold: 0.45
Distribución:
label
1    2618
0    1218
Name: count, dtype: int64

Archivo: submission_sgd_hashing_threshold_048.csv
Threshold: 0.48
Distribución:
label
1    2376
0    1460
Name: count, dtype: int64

Archivo: submission_sgd_hashing_threshold_0495.csv
Threshold: 0.495
Distribución:
label
1    2244
0    1592
Name: count, dtype: int64

Archivo: submission_sgd_hashing_threshold_05.csv
Threshold: 0.5
Distribución:
label
1    2196
0    1640
Name: count, dtype: int64

Archivo: submission_sgd_hashing_threshold_052.csv
Threshold: 0.52
Distribución:
label
1    2018
0    1818
Name: count, dtype: int64

Archivo: submission_sgd_hashing_threshold_055.csv
Threshold: 0.55
Distribución:
label
0    2063
1    1773
Name: count, dtype: int64


## Resumen final

El modelo `SGDClassifier` con `loss="log_loss"` se entrenó utilizando `HashingVectorizer` con n-gramas de palabras y caracteres. Esta representación permitió construir un modelo lineal eficiente en memoria.

Se aplicó `RandomizedSearchCV` para ajustar hiperparámetros como `alpha`, `penalty`, `l1_ratio`, `learning_rate`, `eta0` y `max_iter`. Además, se utilizó `early_stopping` y ajuste del threshold para maximizar F1 macro.

Este modelo sirvió como enfoque lineal avanzado y como comparación frente al modelo basado en CatBoost y embeddings.